# Forecast Using the PatchTST-FM Family


TimeCopilot ships a single `PatchTSTFM` class for IBM's PatchTST time series foundation models. Pass the Hugging Face `repo_id` to select the checkpoint:

| Release | `repo_id` | License |
|---------|-----------|--------|
| **PatchTST-FM r1** (default) | `ibm-research/patchtst-fm-r1` | [CC-BY-NC-SA-4.0](https://huggingface.co/ibm-research/patchtst-fm-r1) |
| **Granite PatchTST-FM r1** | `ibm-granite/granite-timeseries-patchtst-fm-r1` | See model card on Hugging Face |

*Requirements*:

    - Python 3.11 – 3.13
    - CUDA recommended (CPU works but is slower)

PatchTST-FM supports flexible context lengths (default 8,192). We use `context_length=2048` here for faster comparison. For probabilistic forecasts, pass `level` or `quantiles` (quantiles from 0.01 to 0.99).


## Import libraries


In [ ]:
import sys

import pandas as pd

from timecopilot import TimeCopilotForecaster

if sys.version_info < (3, 11) or sys.version_info >= (3, 14):
    raise RuntimeError("PatchTST-FM requires Python >= 3.11 and < 3.14")


## Load the dataset

The DataFrame must include at least the following columns:
- unique_id: Unique identifier for each time series (string)
- ds: Date column (datetime format)
- y: Target variable for forecasting (float format)


In [ ]:
df = pd.read_csv(
    "https://timecopilot.s3.amazonaws.com/public/data/events_pageviews.csv",
    parse_dates=["ds"],
)
df.head()


## Plot the data


In [ ]:
TimeCopilotForecaster.plot(df)


## Create a TimeCopilotForecaster

We compare IBM Research and Granite PatchTST-FM r1 checkpoints. Each model gets a distinct `alias` so forecasts are easy to identify.


In [ ]:
from timecopilot.models.foundation.patchtst_fm import PatchTSTFM

context_length = 2048
batch_size = 32

models = [
    PatchTSTFM(
        repo_id="ibm-research/patchtst-fm-r1",
        alias="PatchTST-FM-r1",
        context_length=context_length,
        batch_size=batch_size,
    ),
    PatchTSTFM(
        repo_id="ibm-granite/granite-timeseries-patchtst-fm-r1",
        alias="Granite-PatchTST-FM-r1",
        context_length=context_length,
        batch_size=batch_size,
    ),
]

tcf = TimeCopilotForecaster(models=models)


## Cross-validate with prediction intervals

Optional parameters:
- `freq`: pandas frequency alias (inferred from `ds` if omitted)
- `h`: forecast horizon
- `level`: prediction interval levels (percent)


In [ ]:
level = [80, 95]
h = 12
cv_df = tcf.cross_validation(df=df, h=h, level=level)
cv_df.head()


## Plot cross-validation forecasts


In [ ]:
tcf.plot(df, cv_df.drop(columns=["cutoff", "y"]), level=[95])


## Evaluate


In [ ]:
from functools import partial

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mase, scaled_crps

metrics_df = evaluate(
    cv_df,
    metrics=[partial(mase, seasonality=12), scaled_crps],
    train_df=df,
)
metrics_df.groupby("metric").mean(numeric_only=True).sort_values(
    by="mase"
)
